In [ ]:
import torch 
import numpy as np

def initialize_population(pop_size, length):
    return [torch.randint(0, 3, (length,)) for _ in range(pop_size)]

def mutate(individual, mutation_rate=0.05):
    new = individual.clone()
    for i in range(len(new)):
        if torch.rand(1).item() < mutation_rate:
            new[i] = torch.randint(0, 3, (1,)).item()
    return new

def crossover(parent1, parent2):
    point = torch.randint(1, len(parent1)-1, (1,)).item()
    child = torch.cat([parent1[:point], parent2[point:]])
    return child

def select_top(population, scores, top_k):
    sorted_indices = np.argsort(scores)[::-1]
    return [population[i] for i in sorted_indices[:top_k]]

def evaluate_schedule(schedule_96, df_test, idx_start, model_temp, model_rh, scaler_temp, scaler_rh):
    schedule = np.array(schedule_96, dtype=int)

    df_test_copy = df_test.copy()
    df_test_copy.loc[df_test_copy.index[idx_start:idx_start+96], 'hvac_mode'] = schedule

    preds_temp = predict_Tin_sequence_full(
        df_test_copy, 
        window_start_idx=idx_start, 
        lstm_model=model_temp['lstm'], 
        rf_model=model_temp['rf'], 
        scaler=model_temp['scaler'],
        window_size=6,
        n_steps=96
    )

    preds_rh = predict_RH_sequence_full(  
        df_test_copy, 
        window_start_idx=idx_start, 
        lstm_model=model_rh['lstm'], 
        rf_model=model_rh['rf'], 
        scaler=model_rh['scaler'],
        window_size=6,
        n_steps=96
    )

    tin_series = preds_temp['Tin_pred'].values
    rh_series = preds_rh['RH_pred'].values
    return compute_simple_comfort(tin_series, rh_series)

def optimize_hvac_schedule(
    df_test,
    idx_start,
    model_temp,
    model_rh,
    pop_size=30,
    generations=30,
    top_k=10,
    mutation_rate=0.1
):
    population = initialize_population(pop_size, 96)

    for gen in range(generations):
        scores = []
        for ind in population:
            hvac_schedule = ind.tolist()
            score = evaluate_schedule(hvac_schedule, df_test, idx_start, model_temp, model_rh)
            scores.append(score)

        print(f"Gen {gen} - Best comfort score: {np.max(scores):.2f}")

        elites = select_top(population, scores, top_k)

        new_population = []
        while len(new_population) < pop_size:
            parents = np.random.choice(elites, size=2, replace=False)
            child = crossover(parents[0], parents[1])
            child = mutate(child, mutation_rate)
            new_population.append(child)

        population = new_population

    best_idx = np.argmax(scores)
    return population[best_idx].tolist()

In [ ]:
best_schedule = optimize_hvac_schedule(
    df_test=df,               # your full dataset
    idx_start=5000,           # start index of the 96-step forecast
    model_temp={
        'lstm': my_temp_lstm,
        'rf': my_temp_rf,
        'scaler': my_temp_scaler
    },
    model_rh={
        'lstm': my_rh_lstm,
        'rf': my_rh_rf,
        'scaler': my_rh_scaler
    }
)

print("Best HVAC schedule (0=OFF, 1=LOW, 2=HIGH):")
print(best_schedule)